In [1]:
import pandas as pd 
import torch 
import torch.nn as nn 

In [2]:
df = pd.read_csv ( '/home/dung/machinelearning/reservoir computing/Mackey-Glass Time Series(taw17).csv')

In [3]:
df.drop ( columns= 'index',inplace = True)

In [9]:
df.shape 

(1100, 3)

In [ ]:
num_input = df.shape[0]
train_size = int ( 0.8 * num_input)
test_size = num_input- train_size 
train_data = df [:train_size ]
test_data = df [ test_size : ]
train_data_input = tr


In [13]:
train_data.shape 

(880, 3)

In [5]:
def train_reservoir_model(model, data, targets, epochs=100, lr=0.01, washout=100):
        """
        Hàm huấn luyện lớp W_out (nn.Linear) bằng Adam Optimizer và MSE Loss
        
        Tham số:
        - model: Instance của lớp Reservoir_computing bạn đã viết
        - data: Tensor đầu vào shape (1110, 3)
        - targets: Tensor nhãn thực tế mong muốn dự báo, ví dụ shape (1110, 1)
        - epochs: Số vòng lặp huấn luyện (100)
        - lr: Learning rate của bộ tối ưu Adam
        - washout: Số bước khởi đầu chuỗi cần bỏ qua để trạng thái mạng ổn định (100 bước)
        """
        
        # 1. Định nghĩa Loss function (Mean Squared Error)
        criterion = nn.MSELoss()
        
        # 2. Định nghĩa Optimizer (Adam) - CHỈ cập nhật các tham số của lớp W_out
        optimizer = torch.optim.Adam(model.W_out.parameters(), lr=lr)
        
        print("--- BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN LỚP W_OUT ---")
        
        for epoch in range(epochs):
            model.train()         # Đưa mô hình về chế độ huấn luyện
            optimizer.zero_grad() # Xóa các gradient từ bước tính trước
            
            # Bước forward: Tính toán dự báo cho toàn bộ chuỗi 1110 bước
            # Y_pred trả về shape (1110, output_dim)
            Y_pred = model(data)
            
            # Chỉ tính toán Loss trên khoảng thời gian SAU washout để triệt tiêu nhiễu ban đầu
            loss = criterion(Y_pred[washout:], targets[washout:])
            
            # Lan truyền ngược (Tính đạo hàm riêng cho lớp W_out)
            loss.backward()
            
            # Cập nhật trọng số của lớp W_out bằng thuật toán Adam
            optimizer.step()
            
            # In tiến trình sau mỗi 10 epochs
            if (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1:3d}/{epochs}], Loss (MSE): {loss.item():.6f}")
                
        print("---------------------------------------------")
        print(" Huấn luyện hoàn tất 100 Epochs!")

In [7]:
from reservoir_computing import Reservoir_computing

if __name__ == "__main__":
    T_steps = 1110         
    input_dim = 3          
    reservoir_dim = 500    
    output_dim = 1         
    leaking_rate = 0.3     
    spectral_radius = 0.95 
    
    print("--- 1. Khởi tạo mô hình Reservoir Computing ---")
    model = Reservoir_computing(
        input_dim=input_dim,
        num_input=T_steps,
        output_dim=output_dim,
        learning_rate=leaking_rate,
        reservoir_dim=reservoir_dim,
        spectral_radius=spectral_radius
    )
    print("Khởi tạo thành công!")
    print("-" * 50)

    print("--- 2. Tạo dữ liệu rác (Shape 1110, 3) và nhãn mục tiêu ---")
    dummy_data = torch.randn(T_steps, input_dim, dtype=torch.float32)
    dummy_targets = torch.randn(T_steps, output_dim, dtype=torch.float32)
    print(f"Shape dữ liệu đầu vào: {dummy_data.shape}")
    print(f"Shape nhãn mục tiêu:  {dummy_targets.shape}")
    print("-" * 50)

    # Tiến hành gọi hàm train
    train_reservoir_model(model, dummy_data, dummy_targets, epochs=100, lr=0.01, washout=100)

--- 1. Khởi tạo mô hình Reservoir Computing ---
Khởi tạo thành công!
--------------------------------------------------
--- 2. Tạo dữ liệu rác (Shape 1110, 3) và nhãn mục tiêu ---
Shape dữ liệu đầu vào: torch.Size([1110, 3])
Shape nhãn mục tiêu:  torch.Size([1110, 1])
--------------------------------------------------
--- BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN LỚP W_OUT ---


AttributeError: 'Reservoir_computing' object has no attribute 'train'

In [8]:
def train_reservoir_model(model, data, targets, epochs=100, lr=0.01, washout=100):
    """
    Hàm huấn luyện lớp W_out tối ưu cho Reservoir Computing
    """
    # 1. Định nghĩa Loss function và Optimizer
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.W_out.parameters(), lr=lr)
    
    print("--- 1. Trích xuất đặc trưng phi tuyến từ Reservoir (Chạy 1 lần duy nhất) ---")
    
    # Sử dụng torch.no_grad() để đóng băng đồ thị tính toán của vòng lặp thời gian
    with torch.no_grad():
        T = data.shape[0]
        current_state = torch.zeros(model.reservoir_dim, dtype=torch.float32)
        states_history = []
        
        # Lan truyền trạng thái qua chuỗi thời gian
        for t in range(T): 
            u_t = data[t] 
            input_part = torch.matmul(model.W_in, u_t)
            reservoir_part = torch.matmul(model.Wrc, current_state)
            new_state = (1 - model.learning_rate) * current_state + model.learning_rate * torch.tanh(input_part + reservoir_part)
            current_state = new_state
            states_history.append(current_state)
            
        X_states = torch.stack(states_history)
        # Tạo ma trận đặc trưng mở rộng cố định
        X_augmented = torch.cat([X_states, data], dim=1)
        
    # Loại bỏ khoảng washout để triệt tiêu nhiễu khởi tạo
    X_train = X_augmented[washout:]  # Cố định đặc trưng đầu vào cho lớp tuyến tính
    Y_train = targets[washout:]      # Cố định nhãn mục tiêu
    
    print(f"-> Ma trận đặc trưng huấn luyện đã sẵn sàng: {X_train.shape}")
    print("--- 2. Bắt đầu tối ưu hóa lớp W_out bằng Adam ---")
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        
        # Lúc này, bước forward diễn ra siêu tốc vì chỉ là một phép nhân ma trận tuyến tính đơn giản
        Y_pred = model.W_out(X_train)
        
        # Tính toán loss trực tiếp
        loss = criterion(Y_pred, Y_train)
        
        # Lan truyền ngược (Đồ thị tính toán bây giờ chỉ sâu đúng 1 tầng của lớp Linear)
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1:3d}/{epochs}], Loss (MSE): {loss.item():.6f}")
                
    print("---------------------------------------------")
    print("✅ Huấn luyện hoàn tất 100 Epochs theo đúng cơ chế Reservoir Computing!")